<a href="https://colab.research.google.com/github/thuynguyenhuit/hocsau/blob/main/TrashNet_Project/Phan_loai_rac_thai_bang_mo_hinh_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1. Kết nối dữ liệu trên drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
data_path = '/content/drive/MyDrive/CNN/Garbage_Data'
if os.path.exists(data_path):
    print("Kết nối thành công! Các lớp rác thải tìm thấy:", os.listdir(data_path))
else:
    print("Đường dẫn chưa đúng, bạn hãy kiểm tra lại tên folder trên Drive.")

Kết nối thành công! Các lớp rác thải tìm thấy: ['trash', 'cardboard', 'metal', 'plastic', 'glass', 'paper']


In [3]:
# import các thư viện
import os
import warnings; warnings.filterwarnings("ignore")
# ── Image Processing ─────────────────────────
import cv2 # read/resize images
from tqdm import tqdm # progress bar
# ── Numerical Computing ──────────────────────
import numpy as np
import pandas as pd
# ── Visualisation ────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# ── Deep Learning ────────────────────────────
import tensorflow as tf
import keras
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image as keras_image
# ── Evaluation ───────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import shuffle


In [4]:
# 2. Cấu hình nhãn (Cập nhật đúng tên folder trong Garbage_Data của bạn)
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
class_names_label = {c: i for i, c in enumerate(class_names)}
Image_Size = (150, 150)

def Load_Data():
    # 3. Cập nhật đường dẫn đến thư mục trên Drive của bạn
    # Dựa trên hình image_87af4c.png, đường dẫn của bạn là:
    dataset = '/content/drive/MyDrive/CNN/Garbage_Data'

    Images, Labels = [], []

    # Duyệt qua từng thư mục con (tương ứng với từng lớp)
    for folder in os.listdir(dataset):
        label = class_names_label.get(folder)
        if label is None: continue # Bỏ qua nếu folder không nằm trong danh sách lớp

        folder_path = os.path.join(dataset, folder)

        # tqdm giúp hiển thị thanh tiến trình khi load ảnh
        for file in tqdm(os.listdir(folder_path), desc=f"Loading {folder}"):
            img_path = os.path.join(folder_path, file)

            try:
                img = cv2.imread(img_path)
                if img is None: continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, Image_Size)

                Images.append(img)
                Labels.append(label)
            except Exception as e:
                print(f"Lỗi khi đọc file {file}: {e}")

    return np.array(Images, dtype='float32'), np.array(Labels, dtype='int32')

# 4. Thực thi load dữ liệu
train_image, train_label = Load_Data()

# 5. Shuffle dữ liệu (Cực kỳ quan trọng để mô hình học khách quan)
train_image, train_label = shuffle(train_image, train_label, random_state=42)

# 6. Chuẩn hóa dữ liệu (Đưa về đoạn [0, 1])
train_image = train_image / 255.0

Loading paper: 100%|██████████| 594/594 [00:13<00:00, 42.64it/s] 
